### در این مثال، مدل سازی پویا برای پیش بینی قیمت نفت برنت آمریکا انجام می شود. این مدلسازی با دو الگوریتم LSTM و ARIMA انجام می شود.
### داده های قیمت نفت برنت در بازه 17 می 1987 تا 13 نوامبر 2022 جمع آوری شده است. پس از مدل سازی پویا، داده های قیمت نفت برنت از یاهو فاینانس بصورت آنلاین دریافت می شود و نتایج مدل مقایسه می شود.
برای شروع ابتدا داده های قیمت نفت برنت از آدرس زیر دانلود و در پوشه کاری قرار داده شود.
https://github.com/hojatollahgholami/AI-in-Oil-and-gas-industry/blob/main/ex3-3/BrentOilPrices.csv

## نصب کتابخانه های موردنیاز

In [1]:
!pip install pmdarima statsmodels arabic-reshaper python-bidi torch 

## فراخوانی کتابخانه های موردنیاز

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import warnings
warnings.filterwarnings('ignore')

# برای نمایش فارسی در نمودارها
import arabic_reshaper
from bidi.algorithm import get_display

# PyTorch کتابخانه های
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from datetime import datetime, timedelta

## تعریف پارامترهای ورودی

In [3]:
# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)


# Create output directory
OUTPUT_DIR = 'oilPrice_dynamic_modeling'
os.makedirs(OUTPUT_DIR, exist_ok=True)

### بارگزاری داده ها و پیش پردازش

In [4]:
# بارگزاری داده
df = pd.read_csv('BrentOilPrices.csv')


# تبدیل ستون زمان رشته ای به زمان قابل فهم سیستم
def convert_date(date_str):
    try:
        return pd.to_datetime(date_str, format='%d-%b-%y')
    except:
        return pd.to_datetime(date_str)


df['Date'] = df['Date'].apply(convert_date)
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

# مدیریت داده های گمشده
df.fillna(method='ffill', inplace=True) 


#  داده ویرایش شده
original_data = df.copy()
print(f"محدوده داده: {df.index.min()} to {df.index.max()}")
original_data

محدوده داده: 1987-05-20 00:00:00 to 2021-12-31 00:00:00


,Price
Date,
1987-05-20,18.63
1987-05-21,18.45
1987-05-22,18.55
1987-05-25,18.60
1987-05-26,18.63
...,...
2021-12-23,76.26
2021-12-24,75.24
2021-12-29,78.63


# بارگیری داده های جدید 

In [5]:
new_data = pd.read_csv('BrentOilPricesUpdate.csv')

# تبدیل ستون زمان رشته ای به زمان قابل فهم سیستم
def convert_date(date_str):
    try:
        return pd.to_datetime(date_str, format='%d-%b-%y')
    except:
        return pd.to_datetime(date_str)


new_data['Date'] = new_data['Date'].apply(convert_date)
new_data.set_index('Date', inplace=True)
new_data.sort_index(inplace=True)

print(f"محدوده داده: {new_data.index.min()} to {new_data.index.max()}")
new_data

محدوده داده: 2022-01-03 00:00:00 to 2022-11-14 00:00:00


,Price
Date,
2022-01-03,78.25
2022-01-04,79.39
2022-01-05,80.60
2022-01-06,81.99
2022-01-07,82.28
...,...
2022-11-08,96.85
2022-11-09,93.05
2022-11-10,94.25


# آماده سازی داده اصلی برای مدل سازی

In [6]:
cutoff_date = pd.Timestamp('2022-11-30')
train_data_original = df[df.index <= cutoff_date]
test_data_original = new_data  # استفاده از داده های جدید به عنوان داده تست


print(f"Training data size: {len(train_data_original)}")
print(f"Test data size: {len(test_data_original)}")

Training data size: 8791
Test data size: 220


In [7]:
# استانداردسازی داده اصلی جهت مدلسازی ARIMA
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_train = scaler.fit_transform(train_data_original.values.reshape(-1, 1))

# مدلسازی ARIMA, LSTM با داده های اصلی

In [8]:
#   استفاده از سری زمانی به جای DataFrame
arima_model = ARIMA(train_data_original['Price'], order=(5,1,0))
arima_model_fit = arima_model.fit()


# پیش بینی با ARIMA
arima_forecast = arima_model_fit.forecast(steps=len(test_data_original))
print("آموزش مدل ARIMA با داده های اصلی انجام شد")

آموزش مدل ARIMA با داده های اصلی...


In [9]:
# مدلسازی با LSTM
# LSTM with PyTorch - مدل بهبود یافته
class ImprovedLSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_layer_size=100, output_size=1, num_layers=3, dropout_rate=0.3):
        super().__init__()
        self.hidden_layer_size = hidden_layer_size
        self.num_layers = num_layers
        
        # LSTM layers با دراپ اوت
        self.lstm = nn.LSTM(
            input_size, 
            hidden_layer_size, 
            num_layers=num_layers, 
            batch_first=True, 
            dropout=dropout_rate if num_layers > 1 else 0
        )
        
        # لایه های کاملاً متصل
        self.fc_layers = nn.Sequential(
            nn.Linear(hidden_layer_size, 50),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(50, 25),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(25, output_size)
        )
        
    def forward(self, input_seq):
        # LSTM forward
        lstm_out, _ = self.lstm(input_seq)
        
        # فقط خروجی آخرین مرحله
        last_output = lstm_out[:, -1, :]
        
        # لایه های کامل متصل
        predictions = self.fc_layers(last_output)
        
        return predictions


# آماده سازی داده برای LSTM
def create_dataset(dataset, time_step=60):
    X, y = [], []
    for i in range(len(dataset)-time_step-1):
        a = dataset[i:(i+time_step), 0]
        X.append(a)
        y.append(dataset[i + time_step, 0])
    return np.array(X), np.array(y)


time_step = 60
X_train, y_train = create_dataset(scaled_train, time_step)


# تبدیل به تانسور PyTorch
X_train_tensor = torch.from_numpy(X_train).float().unsqueeze(2)  # شکل: (samples, time_step, 1)
y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)  # شکل: (samples, 1)


# ایجاد DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)


# پیش آماده سازی مدل, loss function and optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ImprovedLSTMModel().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)


# افزودن برنامه ریزی نرخ آموزش
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=True)


# آموزش مدل
epochs = 100  # افزایش تعداد epoch
train_losses = []
val_losses = []


print("آموزش مدل LSTM با داده های اصلی...")
for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        
        # Gradient clipping برای پایداری آموزش
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
    
    # اعتبارسنجی
    model.eval()
    with torch.no_grad():
        val_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            val_loss += criterion(outputs, batch_y).item()
    
    avg_train_loss = epoch_loss / len(train_loader)
    avg_val_loss = val_loss / len(train_loader)
    
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    
    # Update learning rate
    scheduler.step(avg_val_loss)
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}')


# پیش بینی با مدل LSTM
model.eval()


# برای پیش بینی LSTM، نیاز به ایجاد بازه های کوتاه از انتهای داده های آموزشی
last_sequence = scaled_train[-time_step:]
lstm_forecast = []


with torch.no_grad():
    for i in range(len(test_data_original)):
        seq_tensor = torch.from_numpy(last_sequence).float().unsqueeze(0).to(device)  # Shape: (1, time_step, 1)
        prediction = model(seq_tensor).cpu().numpy()
        lstm_forecast.append(prediction[0, 0])
        # به روزرسانی بازه های کوتاه با پیش بینی جدید
        last_sequence = np.append(last_sequence[1:], prediction[0, 0]).reshape(-1, 1)


# معکوس سازی transform LSTM forecast
lstm_forecast = scaler.inverse_transform(np.array(lstm_forecast).reshape(-1, 1)).flatten()

آموزش مدل LSTM با داده های اصلی...
Epoch [10/100], Train Loss: 0.003500, Val Loss: 0.000637
Epoch [20/100], Train Loss: 0.003086, Val Loss: 0.001107
Epoch [30/100], Train Loss: 0.002919, Val Loss: 0.000452
Epoch [40/100], Train Loss: 0.003044, Val Loss: 0.000349


KeyboardInterrupt: 

# ارزیابی

In [ ]:
arima_rmse_original = np.sqrt(mean_squared_error(test_data_original, arima_forecast))
arima_mae_original = mean_absolute_error(test_data_original, arima_forecast)
arima_mape_original = mean_absolute_percentage_error(test_data_original, arima_forecast)


lstm_rmse_original = np.sqrt(mean_squared_error(test_data_original, lstm_forecast))
lstm_mae_original = mean_absolute_error(test_data_original, lstm_forecast)
lstm_mape_original = mean_absolute_percentage_error(test_data_original, lstm_forecast)


print("\nعملکرد مدل اصلی با داده های جدید:")
print(f"ARIMA - RMSE: {arima_rmse_original:.4f}, MAE: {arima_mae_original:.4f}, MAPE: {arima_mape_original:.4%}")
print(f"LSTM - RMSE: {lstm_rmse_original:.4f}, MAE: {lstm_mae_original:.4f}, MAPE: {lstm_mape_original:.4%}")

# آموزش مجدد مدل ها

In [ ]:
# ترکیب تمام داده ها و آموزش مجدد مدل ها
print("\nترکیب تمام داده ها و آموزش مجدد مدل ها...")
all_data = pd.concat([df, new_data])
all_data = all_data[~all_data.index.duplicated(keep='first')]
all_data.sort_index(inplace=True)


# فقط نگه داشتن ستون قیمت و مدیریت مقادیر NaN
all_data = all_data[['Price']]
all_data.fillna(method='ffill', inplace=True)
all_data.fillna(method='bfill', inplace=True)


print('NaN values after cleaning:', all_data.isna().sum())


# آماده سازی داده برای آموزش کامل
scaler_full = MinMaxScaler(feature_range=(0, 1))
scaled_all = scaler_full.fit_transform(all_data.values.reshape(-1, 1))


# مدیریت مقادیر NaN در داده‌های scaled
scaled_all = np.nan_to_num(scaled_all, nan=0.0, posinf=1.0, neginf=0.0)

In [ ]:
# آموزش مجدد ARIMA روی تمام داده ها
print("آموزش مجدد ARIMA روی تمام داده ها...")
arima_model_full = ARIMA(all_data['Price'], order=(5,1,0))
arima_model_fit_full = arima_model_full.fit()


# For ARIMA forecasting, we need to predict the same test period
# Get the position of the test period in the full dataset
test_start_date = test_data_original.index[0]
test_end_date = test_data_original.index[-1]


# Get the index positions for the test period
test_start_idx = all_data.index.get_loc(test_start_date)
test_end_idx = all_data.index.get_loc(test_end_date)


# Ensure we have valid indices
if test_start_idx >= test_end_idx:
    # If indices are not valid, use the length of test data
    test_start_idx = len(all_data) - len(test_data_original)
    test_end_idx = len(all_data) - 1


# Forecast with the updated ARIMA model
arima_forecast_full = arima_model_fit_full.forecast(steps=test_end_idx - test_start_idx + 1)

In [ ]:
# Retrain LSTM on all data
X_full, y_full = create_dataset(scaled_all, time_step)


# Convert to PyTorch tensors
X_full_tensor = torch.from_numpy(X_full).float().unsqueeze(2)  # Shape: (samples, time_step, 1)
y_full_tensor = torch.from_numpy(y_full).float().unsqueeze(1)  # Shape: (samples, 1)


# Create DataLoader for full data
full_dataset = TensorDataset(X_full_tensor, y_full_tensor)
full_loader = DataLoader(full_dataset, batch_size=64, shuffle=True)


# Train a new LSTM model on all data
new_model = ImprovedLSTMModel().to(device)
new_optimizer = torch.optim.Adam(new_model.parameters(), lr=0.001, weight_decay=1e-5)
new_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(new_optimizer, patience=5, factor=0.5, verbose=True)


new_train_losses = []
new_val_losses = []


print("Retraining LSTM on all data...")
for epoch in range(epochs):
    new_model.train()
    epoch_loss = 0
    
    for batch_x, batch_y in full_loader:
        if torch.isnan(batch_x).any() or torch.isnan(batch_y).any():
            print('NaN detected in batch, skipping...')
            continue
            
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        new_optimizer.zero_grad()
        outputs = new_model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(new_model.parameters(), max_norm=1.0)
        new_optimizer.step()
        
        epoch_loss += loss.item()
    
    # Validation
    new_model.eval()
    with torch.no_grad():
        val_loss = 0
        for batch_x, batch_y in full_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = new_model(batch_x)
            val_loss += criterion(outputs, batch_y).item()
    
    avg_train_loss = epoch_loss / len(full_loader)
    avg_val_loss = val_loss / len(full_loader)
    
    new_train_losses.append(avg_train_loss)
    new_val_losses.append(avg_val_loss)
    
    # Update learning rate
    new_scheduler.step(avg_val_loss)
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}')


# For LSTM - we need to forecast using the new model
# Find the starting sequence for the test period
test_start_pos = len(all_data) - len(test_data_original) - time_step
if test_start_pos < 0:
    test_start_pos = 0


last_sequence_full = scaled_all[test_start_pos:test_start_pos+time_step]
lstm_forecast_full = []


new_model.eval()
with torch.no_grad():
    for i in range(len(test_data_original)):
        seq_tensor = torch.from_numpy(last_sequence_full).float().unsqueeze(0).to(device)  # Shape: (1, time_step, 1)
        prediction = new_model(seq_tensor).cpu().numpy()
        lstm_forecast_full.append(prediction[0, 0])
        # Update the sequence with the new prediction
        last_sequence_full = np.append(last_sequence_full[1:], prediction[0, 0]).reshape(-1, 1)


# Inverse transform LSTM forecast
lstm_forecast_full = scaler_full.inverse_transform(np.array(lstm_forecast_full).reshape(-1, 1)).flatten()

In [ ]:
# Calculate metrics for updated models
arima_rmse_updated = np.sqrt(mean_squared_error(test_data_original, arima_forecast_full))
arima_mae_updated = mean_absolute_error(test_data_original, arima_forecast_full)
arima_mape_updated = mean_absolute_percentage_error(test_data_original, arima_forecast_full)


lstm_rmse_updated = np.sqrt(mean_squared_error(test_data_original, lstm_forecast_full))
lstm_mae_updated = mean_absolute_error(test_data_original, lstm_forecast_full)
lstm_mape_updated = mean_absolute_percentage_error(test_data_original, lstm_forecast_full)


print("\nPerformance of updated models on new data:")
print(f"ARIMA - RMSE: {arima_rmse_updated:.4f}, MAE: {arima_mae_updated:.4f}, MAPE: {arima_mape_updated:.4%}")
print(f"LSTM - RMSE: {lstm_rmse_updated:.4f}, MAE: {lstm_mae_updated:.4f}, MAPE: {lstm_mape_updated:.4%}")


# Calculate improvement
arima_rmse_improvement = (arima_rmse_original - arima_rmse_updated) / arima_rmse_original * 100
lstm_rmse_improvement = (lstm_rmse_original - lstm_rmse_updated) / lstm_rmse_original * 100


print(f"\nImprovement after updating models:")
print(f"ARIMA RMSE improvement: {arima_rmse_improvement:.2f}%")
print(f"LSTM RMSE improvement: {lstm_rmse_improvement:.2f}%")

# مصورسازی

In [ ]:
# Create comprehensive comparison plots
plt.figure(figsize=(16, 12))


# Plot 1: Actual vs Predicted for both models
plt.subplot(2, 2, 1)
plt.plot(test_data_original.index, test_data_original.values, label='داده واقعی', color='black', linewidth=2)
plt.plot(test_data_original.index, arima_forecast, label='پیش‌بینی ARIMA (مدل اولیه)', color='red', linestyle='--')
plt.plot(test_data_original.index, arima_forecast_full, label='پیش‌بینی ARIMA (مدل به‌روز شده)', color='blue', linestyle='--')
plt.title(get_display(arabic_reshaper.reshape('مقایسه پیش‌بینی‌های ARIMA')))
plt.ylabel(get_display(arabic_reshaper.reshape('قیمت (دلار)')))
plt.xlabel(get_display(arabic_reshaper.reshape('تاریخ')))
plt.legend()
plt.grid(True)


plt.subplot(2, 2, 2)
plt.plot(test_data_original.index, test_data_original.values, label='داده واقعی', color='black', linewidth=2)
plt.plot(test_data_original.index, lstm_forecast, label='پیش‌بینی LSTM (مدل اولیه)', color='red', linestyle='--')
plt.plot(test_data_original.index, lstm_forecast_full, label='پیش‌بینی LSTM (مدل به‌روز شده)', color='blue', linestyle='--')
plt.title(get_display(arabic_reshaper.reshape('مقایسه پیش‌بینی‌های LSTM')))
plt.ylabel(get_display(arabic_reshaper.reshape('قیمت (دلار)')))
plt.xlabel(get_display(arabic_reshaper.reshape('تاریخ')))
plt.legend()
plt.grid(True)

# Plot 2: Error comparison
plt.subplot(2, 2, 3)
models = ['ARIMA اولیه', 'ARIMA به‌روز', 'LSTM اولیه', 'LSTM به‌روز']
rmse_values = [arima_rmse_original, arima_rmse_updated, lstm_rmse_original, lstm_rmse_updated]
plt.bar(models, rmse_values, color=['red', 'lightcoral', 'blue', 'lightblue'])
plt.title(get_display(arabic_reshaper.reshape('مقایسه خطای RMSE مدل‌ها')))
plt.ylabel(get_display(arabic_reshaper.reshape('مقدار RMSE')))
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

# Plot 3: Improvement percentage
plt.subplot(2, 2, 4)
improvement = [0, arima_rmse_improvement, 0, lstm_rmse_improvement]
plt.bar(['ARIMA', 'LSTM'], [arima_rmse_improvement, lstm_rmse_improvement], color=['lightcoral', 'lightblue'])
plt.title(get_display(arabic_reshaper.reshape('درصد بهبود مدل‌ها پس از به‌روزرسانی')))
plt.ylabel(get_display(arabic_reshaper.reshape('درصد بهبود')))
plt.grid(True, alpha=0.3)


plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Create a detailed results table
results_df = pd.DataFrame({
    'Model': ['ARIMA (Original)', 'ARIMA (Updated)', 'LSTM (Original)', 'LSTM (Updated)'],
    'RMSE': [arima_rmse_original, arima_rmse_updated, lstm_rmse_original, lstm_rmse_updated],
    'MAE': [arima_mae_original, arima_mae_updated, lstm_mae_original, lstm_mae_updated],
    'MAPE': [arima_mape_original, arima_mape_updated, lstm_mape_original, lstm_mape_updated],
    'Improvement (%)': [0, arima_rmse_improvement, 0, lstm_rmse_improvement]
})


print("\nنتایج کامل مقایسه مدل‌ها:")
print(results_df.to_string(index=False))


# Save results to CSV
results_df.to_csv(os.path.join(OUTPUT_DIR, 'model_comparison_results.csv'), index=False)


In [ ]:
# Plot training history
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='آموزش مدل اولیه')
plt.plot(val_losses, label='اعتبارسنجی مدل اولیه')
plt.plot(new_train_losses, label='آموزش مدل به‌روز شده')
plt.plot(new_val_losses, label='اعتبارسنجی مدل به‌روز شده')
plt.title(get_display(arabic_reshaper.reshape('تاریخچه آموزش مدل LSTM')))
plt.ylabel(get_display(arabic_reshaper.reshape('خطا')))
plt.xlabel(get_display(arabic_reshaper.reshape('دوره')))
plt.legend()
plt.grid(True)


plt.subplot(1, 2, 2)
plt.plot(test_data_original.index, abs(test_data_original.values - arima_forecast), label='ARIMA اولیه')
plt.plot(test_data_original.index, abs(test_data_original.values - arima_forecast_full), label='ARIMA به‌روز شده')
plt.plot(test_data_original.index, abs(test_data_original.values - lstm_forecast), label='LSTM اولیه')
plt.plot(test_data_original.index, abs(test_data_original.values - lstm_forecast_full), label='LSTM به‌روز شده')
plt.title(get_display(arabic_reshaper.reshape('خطای مطلق پیش‌بینی‌ها')))
plt.ylabel(get_display(arabic_reshaper.reshape('خطای مطلق')))
plt.xlabel(get_display(arabic_reshaper.reshape('تاریخ')))
plt.legend()
plt.grid(True)


plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_error_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()


# نمودارهای اضافی برای تحلیل جامع‌تر
plt.figure(figsize=(15, 10))


# داده‌های تاریخی کامل
plt.subplot(2, 2, 1)
plt.plot(original_data.index, original_data['Price'], label='داده‌های واقعی', color='blue')
plt.axvline(x=cutoff_date, color='red', linestyle='--', label='تاریخ برش (نوامبر ۲۰۲۲)')
plt.title(get_display(arabic_reshaper.reshape('داده‌های تاریخی قیمت نفت')))
plt.ylabel(get_display(arabic_reshaper.reshape('قیمت (دلار)')))
plt.xlabel(get_display(arabic_reshaper.reshape('تاریخ')))
plt.legend()
plt.grid(True)


# داده‌های آموزشی و تست
plt.subplot(2, 2, 2)
plt.plot(train_data_original.index, train_data_original['Price'], label='داده‌های آموزشی', color='green')
plt.plot(test_data_original.index, test_data_original['Price'], label='داده‌های تست', color='orange')
plt.axvline(x=cutoff_date, color='red', linestyle='--', label='تاریخ برش')
plt.title(get_display(arabic_reshaper.reshape('تقسیم‌بندی داده‌های آموزشی و تست')))
plt.ylabel(get_display(arabic_reshaper.reshape('قیمت (دلار)')))
plt.xlabel(get_display(arabic_reshaper.reshape('تاریخ')))
plt.legend()
plt.grid(True)


# مقایسه پیش‌بینی مدل‌های اولیه روی داده‌های اصلی
plt.subplot(2, 2, 3)
# محاسبه پیش‌بینی‌ها روی داده‌های آموزشی
train_arima_predict = arima_model_fit.predict(start=time_step+1, end=len(train_data_original)-1)


# برای LSTM، از آخرین بخش داده‌های آموزشی برای پیش‌بینی استفاده می‌کنیم
last_sequence_train = scaled_train[:time_step]
lstm_train_predict = []


model.eval()
with torch.no_grad():
    for i in range(len(train_data_original) - time_step - 1):
        seq_tensor = torch.from_numpy(last_sequence_train).float().unsqueeze(0).to(device)
        prediction = model(seq_tensor).cpu().numpy()
        lstm_train_predict.append(prediction[0, 0])
        last_sequence_train = np.append(last_sequence_train[1:], scaled_train[time_step + i]).reshape(-1, 1)


# تبدیل به مقیاس اصلی
lstm_train_predict = scaler.inverse_transform(np.array(lstm_train_predict).reshape(-1, 1)).flatten()


plt.plot(train_data_original.index[time_step+1:time_step+1+len(train_arima_predict)], 
         train_data_original['Price'].iloc[time_step+1:time_step+1+len(train_arima_predict)], 
         label='داده واقعی', color='black')
plt.plot(train_data_original.index[time_step+1:time_step+1+len(train_arima_predict)], 
         train_arima_predict, label='پیش‌بینی ARIMA', color='red', linestyle='--')
plt.plot(train_data_original.index[time_step+1:time_step+1+len(lstm_train_predict)], 
         lstm_train_predict, label='پیش‌بینی LSTM', color='blue', linestyle='--')
plt.title(get_display(arabic_reshaper.reshape('پیش‌بینی مدل‌ها روی داده‌های آموزشی')))
plt.ylabel(get_display(arabic_reshaper.reshape('قیمت (دلار)')))
plt.xlabel(get_display(arabic_reshaper.reshape('تاریخ')))
plt.legend()
plt.grid(True)


# خطای مدل‌ها روی داده‌های آموزشی
plt.subplot(2, 2, 4)
arima_train_errors = abs(train_data_original['Price'].iloc[time_step+1:time_step+1+len(train_arima_predict)] - train_arima_predict)
lstm_train_errors = abs(train_data_original['Price'].iloc[time_step+1:time_step+1+len(lstm_train_predict)] - lstm_train_predict)


plt.plot(train_data_original.index[time_step+1:time_step+1+len(train_arima_predict)], 
         arima_train_errors, label='خطای ARIMA', color='red')
plt.plot(train_data_original.index[time_step+1:time_step+1+len(lstm_train_predict)], 
         lstm_train_errors, label='خطای LSTM', color='blue')
plt.title(get_display(arabic_reshaper.reshape('خطای مدل‌ها روی داده‌های آموزشی')))
plt.ylabel(get_display(arabic_reshaper.reshape('خطای مطلق')))
plt.xlabel(get_display(arabic_reshaper.reshape('تاریخ')))
plt.legend()
plt.grid(True)


plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'additional_analysis_1.png'), dpi=300, bbox_inches='tight')
plt.show()


# 2. تحلیل علت خطای بالای مدل بهبود یافته و راهکارهای بهبود
print("\n" + "="*60)
print("تحلیل علل خطای بالای مدل بهبود یافته و راهکارها")
print("="*60)


# محاسبه خطاهای نسبی
print("میانگین خطاهای مطلق:")
print(f"ARIMA اولیه: {arima_mae_original:.4f}")
print(f"ARIMA بهبود یافته: {arima_mae_updated:.4f}")
print(f"LSTM اولیه: {lstm_mae_original:.4f}")
print(f"LSTM بهبود یافته: {lstm_mae_updated:.4f}")


# بررسی تغییرات داده‌ها
price_change = (test_data_original['Price'].iloc[-1] - test_data_original['Price'].iloc[0]) / test_data_original['Price'].iloc[0] * 100
volatility = test_data_original['Price'].std() / test_data_original['Price'].mean() * 100


print(f"\nتغییرات قیمت در دوره تست: {price_change:.2f}%")
print(f"نوسان قیمت در دوره تست: {volatility:.2f}%")


# راهکارهای بهبود مدل
print("\nراهکارهای بهبود مدل:")
print("1. استفاده از داده‌های exogenous (مانند نرخ ارز، تولید نفت اوپک)")
print("2. تنظیم هیپرپارامترهای مدل (بهینه‌سازی دوره look-back)")
print("3. استفاده از معماری‌های پیشرفته‌تر (مانند GRU یا Transformer)")
print("4. اضافه کردن مکانیسم توجه (Attention Mechanism) به LSTM")
print("5. استفاده از ensemble learning ترکیب چند مدل")
print("6. به‌روزرسانی تدریجی مدل با داده‌های جدید به جای بازآموزی کامل")


# 3. پیاده‌سازی راهکارهای بهبود
# استفاده از داده‌های exogenous (نمونه)
print("\nاضافه کردن داده‌های exogenous برای بهبود مدل...")


def download_exogenous_data():
    """دانلود داده‌های exogenous برای بهبود مدل"""
    try:
        # نرخ دلار
        usd_data = yf.download('USDIRR=X', start=start_date_new, end=end_date_new)
        usd_data = usd_data[['Close']].rename(columns={'Close': 'USD_Rate'})
        
        # شاخص S&P 500 (نماینده وضعیت اقتصاد جهانی)
        sp500_data = yf.download('^GSPC', start=start_date_new, end=end_date_new)
        sp500_data = sp500_data[['Close']].rename(columns={'Close': 'SP500'})
        
        # ترکیب داده‌ها
        exogenous_data = pd.concat([usd_data, sp500_data], axis=1)
        exogenous_data.fillna(method='ffill', inplace=True)
        
        return exogenous_data
    except Exception as e:
        print(f"خطا در دریافت داده‌های exogenous: {e}")
        return None


# دانلود داده‌های exogenous
exogenous_data = download_exogenous_data()
if exogenous_data is not None:
    print("داده‌های exogenous با موفقیت دریافت شدند")
    print(exogenous_data.head())
else:
    print("استفاده از داده‌های مصنوعی برای exogenous")
    # ایجاد داده‌های مصنوعی برای نمایش
    exogenous_data = pd.DataFrame({
        'USD_Rate': np.random.normal(42000, 1000, len(test_data_original)),
        'SP500': np.random.normal(4000, 200, len(test_data_original))
    }, index=test_data_original.index)


# 4. نمودار نهایی مقایسه بهبودها
plt.figure(figsize=(15, 8))


# مقایسه خطاها
models = ['ARIMA اولیه', 'ARIMA بهبودیافته', 'LSTM اولیه', 'LSTM بهبودیافته']
rmse_values = [arima_rmse_original, arima_rmse_updated, lstm_rmse_original, lstm_rmse_updated]
mae_values = [arima_mae_original, arima_mae_updated, lstm_mae_original, lstm_mae_updated]


x = np.arange(len(models))
width = 0.35


plt.subplot(1, 2, 1)
plt.bar(x - width/2, rmse_values, width, label='RMSE', color='lightcoral')
plt.bar(x + width/2, mae_values, width, label='MAE', color='lightblue')
plt.xlabel('مدل‌ها')
plt.ylabel('مقدار خطا')
plt.title(get_display(arabic_reshaper.reshape('مقایسه خطاهای مدل‌های مختلف')))
plt.xticks(x, models, rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)


# درصد بهبود
improvement_arima = max(0, (arima_rmse_original - arima_rmse_updated) / arima_rmse_original * 100)
improvement_lstm = max(0, (lstm_rmse_original - lstm_rmse_updated) / lstm_rmse_original * 100)


plt.subplot(1, 2, 2)
plt.bar(['ARIMA', 'LSTM'], [improvement_arima, improvement_lstm], 
        color=['lightgreen' if x > 0 else 'lightcoral' for x in [improvement_arima, improvement_lstm]])
plt.xlabel('مدل')
plt.ylabel('درصد بهبود')
plt.title(get_display(arabic_reshaper.reshape('درصد بهبود پس از به‌روزرسانی مدل')))
plt.grid(True, alpha=0.3)


# افزودن مقادیر روی نمودار
for i, v in enumerate([improvement_arima, improvement_lstm]):
    plt.text(i, v + 0.5, f'{v:.1f}%', ha='center', va='bottom')


plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'final_comparison_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()


print(f"\nهمه نمودارها و تحلیل‌ها در پوشه '{OUTPUT_DIR}' ذخیره شدند.")
print("تحلیل کامل با موفقیت انجام شد!")